In [1]:
df_m = df_all.copy()
df_m["yr_month"] = df_m["datetime"].dt.to_period("M")
monthly_ts = df_m.groupby(["station", "yr_month"])["AQI"].mean().reset_index()
monthly_ts["ym_dt"] = monthly_ts["yr_month"].dt.to_timestamp()
monthly_mean = df_m.groupby("yr_month")["AQI"].mean().reset_index()
monthly_mean["ym_dt"] = monthly_mean["yr_month"].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(16, 6))
for i, (st, g) in enumerate(monthly_ts.groupby("station")):
    g2 = g.sort_values("ym_dt")
    ax.plot(g2["ym_dt"], g2["AQI"], linewidth=1.3,
            color=PALETTE[i % len(PALETTE)], label=st, alpha=0.8)
ax.plot(monthly_mean["ym_dt"], monthly_mean["AQI"], color="black",
        linewidth=2.5, linestyle="--", label="All-station mean")
for yr in range(2013, 2018):
    ax.axvspan(pd.Timestamp(f"{yr-1}-12-01"), pd.Timestamp(f"{yr}-02-28"),
               alpha=0.07, color="blue")
ax.set_xlabel("Time")
ax.set_ylabel("AQI")
ax.set_ylim(bottom=0)
ax.set_title("Figure 2.2 - AQI Time Trend (2013-2017)", fontsize=13, fontweight="bold")
ax.legend(ncol=3, fontsize=7.5, framealpha=0.7)
winter_p = mpatches.Patch(color="blue", alpha=0.15, label="Winter")
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles + [winter_p], labels + ["Winter"], ncol=3, fontsize=7.5)
plt.tight_layout()
# display(fig)
plt.savefig("fig_2_2_aqi_time_trend.png", dpi=120)
print("Saved fig_2_2_aqi_time_trend.png")

yr_avg = df_all.groupby("year")["AQI"].mean()
print("Average AQI by Year:\n", yr_avg.round(1).to_string())


### Bước 2.3: Xu Hướng AQI Trong Ngày (Diurnal Pattern)

**Code làm gì**: Tính AQI trung bình theo từng giờ (0–23h) theo trạm và mùa.

**Biểu đồ**: Line chart 24h — so sánh diễn biến AQI giữa các trạm và nhận diện các khung giờ có AQI cao trong ngày, đồng thời đối chiếu sự khác biệt theo mùa.


In [ ]:
diurnal = df_all.groupby(["station", "hour"])["AQI"].mean().reset_index()
diurnal_mean = df_all.groupby("hour")["AQI"].mean()
diurnal_season = df_all.groupby(["season", "hour"])["AQI"].mean().reset_index()
season_colors = {"Spring":"#2ca02c", "Summer":"#ff7f0e", "Autumn":"#9467bd", "Winter":"#1f77b4"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for i, (st, g) in enumerate(diurnal.groupby("station")):
    axes[0].plot(g["hour"], g["AQI"], linewidth=1.2,
                 color=PALETTE[i % len(PALETTE)], label=st, alpha=0.85)
axes[0].plot(diurnal_mean.index, diurnal_mean.values, "k--", linewidth=2.5, label="Mean")
axes[0].set_xticks(range(0, 24, 2)); axes[0].legend(ncol=2, fontsize=7.5)
axes[0].set_xlabel("Hour"); axes[0].set_ylabel("AQI")
axes[0].set_title("(a) Hourly AQI - All Stations", fontsize=11, fontweight="bold")

for sn in season_order:
    g = diurnal_season[diurnal_season["season"] == sn]
    axes[1].plot(g["hour"], g["AQI"], linewidth=2.2,
                 color=season_colors[sn], label=sn, marker="o", markersize=3)
axes[1].set_xticks(range(0, 24, 2)); axes[1].legend(fontsize=10)
axes[1].set_xlabel("Hour"); axes[1].set_ylabel("AQI")
axes[1].set_title("(b) Hourly AQI - By Season", fontsize=11, fontweight="bold")

plt.suptitle("Figure 2.3 - Diurnal AQI Pattern", fontsize=13, fontweight="bold")
plt.tight_layout()
# display(fig)
plt.savefig("fig_2_3_diurnal.png", dpi=120)
print("Saved fig_2_3_diurnal.png")
print(f"Peak AQI at hour {int(diurnal_mean.idxmax())}h: {diurnal_mean.max():.1f}")


### Bước 2.4: So Sánh 12 Trạm — Boxplot PM2.5

**Code làm gì**: Boxplot phân bố AQI theo trạm, sắp xếp theo trung vị.

**Biểu đồ**: Phân biệt rõ trạm nội đô (ô nhiễm cao, IQR rộng) và trạm ngoại ô/nền như Dingling, Huairou (trung vị thấp hơn).


In [ ]:
st_medians = df_all.groupby("station")["AQI"].median().sort_values()
st_order   = st_medians.index.tolist()
fig, ax = plt.subplots(figsize=(14, 6))
bp_data = [df_all[df_all["station"]==st]["AQI"].dropna().values for st in st_order]
bp = ax.boxplot(bp_data, patch_artist=True,
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3),
                whiskerprops=dict(linewidth=1.2))
for patch, color in zip(bp["boxes"], [PALETTE[i % 12] for i in range(len(st_order))]):
    patch.set_facecolor(color); patch.set_alpha(0.8)
ax.set_xticks(range(1,len(st_order)+1))
ax.set_xticklabels(st_order, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("AQI (ug/m3)")
ax.axhline(df_all["AQI"].median(), color="red", linestyle="--",
           label=f"Overall median = {df_all['AQI'].median():.1f}")
ax.legend(); ax.set_title("Figure 2.4 - AQI Distribution by Station", fontsize=13, fontweight="bold")
plt.tight_layout()
# display(fig)
plt.savefig("fig_2_4_boxplot.png", dpi=120)
print("Saved fig_2_4_boxplot.png")
print(f"Most polluted: {st_medians.idxmax()} ({st_medians.max():.1f})")
print(f"Cleanest: {st_medians.idxmin()} ({st_medians.min():.1f})")


### Bước 2.5: Ma Trận Tương Quan Ô Nhiễm ↔ Khí Tượng

**Code làm gì**: Tính ma trận tương quan Pearson giữa ô nhiễm và khí tượng.

**Biểu đồ**: Heatmap correlation — quan sát thực tế: PRES tương quan dương (nghịch nhiệt), WSPM tương quan âm (gió phân tán ô nhiễm).


In [ ]:
corr_cols = POLLUTANTS + METEO
corr_df   = df_all[corr_cols].corr()
fig, ax   = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_df, ax=ax, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
            linewidths=0.5, vmin=-1, vmax=1, square=True,
            cbar_kws={"label":"Pearson correlation coefficient"},
            annot_kws={"size": 8})
ax.set_title("Figure 2.5 - Pollutant and Meteorological Correlation Matrix",
             fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
# display(fig)
plt.savefig("fig_2_5_correlation.png", dpi=120)
print("Saved fig_2_5_correlation.png")
pm25_corr = corr_df["PM2.5"].drop("PM2.5").sort_values(key=abs, ascending=False)
print("Correlation with PM2.5:\n", pm25_corr.round(3).to_string())


### Bước 2.6: Phân Tích Hướng Gió & Mức Độ Ô Nhiễm (Wind Rose)

**Code làm gì**: Vẽ Wind Rose thể hiện phân bố tần suất hướng gió và PM2.5 tương ứng.

**Phương pháp**: Dùng thư viện `windrose` (ưu tiên); fallback về polar bar chart matplotlib.

**Biểu đồ**: Wind Rose tô màu theo nồng độ PM2.5 — hướng gió nào kéo theo ô nhiễm cao = hướng truyền tải chính.


In [ ]:
wd_map = {"N":0,"NNE":22.5,"NE":45,"ENE":67.5,"E":90,"ESE":112.5,
          "SE":135,"SSE":157.5,"S":180,"SSW":202.5,"SW":225,"WSW":247.5,
          "W":270,"WNW":292.5,"NW":315,"NNW":337.5}
wr_df = df_all[["wd","WSPM","PM2.5"]].dropna().copy()
wr_df["wd_deg"] = wr_df["wd"].map(wd_map)
wr_df = wr_df.dropna(subset=["wd_deg"])

try:
    from windrose import WindroseAxes
    fig = plt.figure(figsize=(10, 8))
    ax_wr = WindroseAxes.from_ax(fig=fig)
    bins_pm25 = [0, 50, 100, 150, 200, 300, 999]
    ax_wr.bar(wr_df["wd_deg"].values, wr_df["PM2.5"].values,
              bins=bins_pm25, normed=True, opening=0.85, edgecolor="white",
              linewidth=0.4, cmap=plt.cm.RdYlGn_r)
    ax_wr.set_legend(title="PM2.5 (ug/m3)", loc="lower right", fontsize=8)
    ax_wr.set_title("Figure 2.6 - Wind Rose: PM2.5 by Wind Direction",
                    fontsize=13, fontweight="bold", pad=30)
    plt.tight_layout()
    # display(fig)
    plt.savefig("fig_2_6_windrose.png", dpi=120)
    print("Saved fig_2_6_windrose.png (windrose)")
except ImportError:
    # Polar fallback
    dir_pm25 = wr_df.groupby("wd_deg")["PM2.5"].mean()
    dir_freq = wr_df["wd_deg"].value_counts(normalize=True)
    common_dirs = dir_pm25.index.intersection(dir_freq.index)
    angles = np.deg2rad(common_dirs)
    pm25_v = dir_pm25.loc[common_dirs].values
    freq_v = dir_freq.loc[common_dirs].values

    fig = plt.figure(figsize=(9, 8))
    ax_p = fig.add_subplot(111, projection="polar")
    ax_p.bar(angles, freq_v*100, width=np.deg2rad(22.5), bottom=0,
             alpha=0.75, edgecolor="white",
             color=plt.cm.RdYlGn_r(pm25_v / pm25_v.max()))
    ax_p.set_theta_zero_location("N"); ax_p.set_theta_direction(-1)
    sm = plt.cm.ScalarMappable(cmap="RdYlGn_r",
                                norm=plt.Normalize(pm25_v.min(), pm25_v.max()))
    sm.set_array([])
    plt.colorbar(sm, ax=ax_p, shrink=0.6, pad=0.1, label="Average PM2.5")
    ax_p.set_title("Figure 2.6 - Polar Bar: PM2.5 by Wind Direction",
                   fontsize=12, fontweight="bold", pad=30)
    plt.tight_layout()
    # display(fig)
    plt.savefig("fig_2_6_windrose.png", dpi=120)
    print("Saved fig_2_6_windrose.png (polar fallback)")
